# Tableaux et figures du mémoire
## Reconnaissance du genre - noms propres ivoiriens | SIGAP / AFOR
**INP-HB · ESI · Stage 2A · 2025-2026**

Génère, à partir des résultats enregistrés, les éléments à insérer dans le mémoire :

1. Tableau comparatif final (test) et graphique
2. Courbes ROC des modèles finalistes et matrice de confusion du modèle retenu
3. Compromis précision / taux de réponse (mode « incertain »)
4. Courbe d'apprentissage
5. Exemples d'interprétabilité du modèle de production (noms fictifs)
6. Configuration retenue pour le v5 (recherche Optuna)

Les figures sont enregistrées dans `figures/memoire/` (PNG, 200 dpi), les tableaux en CSV.

## 0. Paramètres

In [ ]:
FINAL_DIR   = 'runs/final'
DIAG_JSON   = 'runs/final/diagnostic_synthese.json'
BEST_V5     = 'runs/best_v5.json'
PAQUET_DIR  = 'production'
OUT         = 'figures/memoire'

## 1. Configuration

In [ ]:
import os, re, glob, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
from adaptateurs import charger, clean_text

os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 200, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})
BLEU, ORANGE, GRIS, ROUGE = '#2E75B6', '#F3802F', '#8E9BB5', '#E15759'
NOMS = {'regression_logistique': 'Régression logistique', 'v5': 'Modèle par mot (v5)',
        'v4': 'CNN + BiLSTM + MHA (v4)', 'xgboost_stacking': 'XGBoost (empilement)',
        'lightgbm_stacking': 'LightGBM (empilement)', 'xgboost_caracteristiques': 'XGBoost',
        'lightgbm_caracteristiques': 'LightGBM'}

def joli(nom):
    if nom.startswith('ensemble_lr+'):
        return 'Ensemble rég. log. + ' + NOMS.get(nom[len('ensemble_lr+'):], nom).split(' (')[0]
    return NOMS.get(nom, nom)

def sauver(fig, nom):
    fig.savefig(os.path.join(OUT, nom), bbox_inches='tight'); plt.show()

ev = json.load(open(os.path.join(FINAL_DIR, 'evaluation_finale.json'), encoding='utf-8'))
res = pd.DataFrame(ev['resultats'])
print(f"Modèle retenu : {joli(ev['choix'])} ({ev['raison']})")

## 2. Tableau comparatif final (test)

In [ ]:
tab = res.assign(Modèle=res['modele'].map(joli)).rename(columns={
    'acc': 'Accuracy', 'acc_ic95': 'IC 95 %', 'f1': 'F1 macro', 'auc': 'AUC-ROC',
    'mcnemar_p': 'McNemar p (vs rég. log.)', 'taux_reponse': 'Taux de réponse (seuil 0,70)',
    'precision_si_reponse': 'Précision si réponse'})
tab['Accuracy'] = [f"{a:.4f} ± {s:.4f}" if g > 1 else f"{a:.4f}"
                   for a, s, g in zip(res['acc'], res['acc_std'], res['graines'])]
cols = ['Modèle', 'Accuracy', 'IC 95 %', 'F1 macro', 'AUC-ROC', 'McNemar p (vs rég. log.)',
        'Taux de réponse (seuil 0,70)', 'Précision si réponse']
tab = tab[cols].round(4)
tab.to_csv(os.path.join(OUT, 'tableau_comparatif_test.csv'), index=False, encoding='utf-8-sig')
display(tab)

ordre = res.sort_values('acc')
fig, ax = plt.subplots(figsize=(8, 0.45 * len(ordre) + 1.2))
couleurs = [ROUGE if m == ev['choix'] else (ORANGE if m.startswith('ensemble') else BLEU) for m in ordre['modele']]
ax.barh(ordre['modele'].map(joli), ordre['acc'], xerr=ordre['acc_std'], color=couleurs, capsize=3)
for y, (a, sd) in enumerate(zip(ordre['acc'], ordre['acc_std'])):
    ax.text(a + sd + 0.0012, y, f"{a:.4f}" + (f" ± {sd:.4f}" if sd > 0 else ""), va='center', fontsize=9)
ax.set_xlim(ordre['acc'].min() - 0.02, ordre['acc'].max() + 0.016)
ax.set_xlabel('Accuracy sur le jeu de test (3 887 noms)')
ax.set_title('Comparaison des modèles (rouge : modèle retenu)')
sauver(fig, 'comparaison_test.png')

## 3. Courbes ROC et matrice de confusion

In [ ]:
df = pd.read_csv('dataset_nettoye.csv')
df['full_name'] = (df['NOM'].apply(clean_text) + ' ' + df['PRENOMS'].fillna('').apply(clean_text)).str.strip()
df = df[df['full_name'] != ''].reset_index(drop=True)
sp = pd.read_csv('split_reference.csv')
emp = df['full_name'].map(lambda s: hashlib.sha1(s.encode()).hexdigest()[:10]).values
assert len(sp) == len(df) and (sp['empreinte'].values == emp).all()
y_te = (df.loc[sp['ensemble'].values == 'test', 'GENRE'] == 'M').astype(int).values

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, make_union
tr = sp['ensemble'].values == 'train'
lr = make_pipeline(make_union(TfidfVectorizer(analyzer='char_wb', ngram_range=(1, 6), sublinear_tf=True),
                              TfidfVectorizer(analyzer='word', token_pattern=r'\S+')),
                   LogisticRegression(C=2, max_iter=3000))
lr.fit(df.loc[tr, 'full_name'], (df.loc[tr, 'GENRE'] == 'M').astype(int))
P = {'regression_logistique': lr.predict_proba(df.loc[sp['ensemble'].values == 'test', 'full_name'])[:, 1]}
for arch in ('v5', 'v4'):
    files = glob.glob(os.path.join(FINAL_DIR, f'{arch}_seed*.npz'))
    if files:
        P[arch] = np.mean([np.load(f)['test'] for f in files], axis=0)
for f in glob.glob(os.path.join(FINAL_DIR, 'xgboost_stacking.npz')):
    P['xgboost_stacking'] = np.load(f)['test']

fig, ax = plt.subplots(figsize=(6, 5.5))
ax.plot([0, 1], [0, 1], '--', color=GRIS, lw=1)
for (nom, p), c in zip(P.items(), [ROUGE, BLEU, ORANGE, '#59A14F']):
    fpr, tpr, _ = roc_curve(y_te, p)
    suffixe = ', 3 graines moyennées' if nom in ('v4', 'v5') else ''
    ax.plot(fpr, tpr, color=c, lw=2 if nom == ev['choix'] else 1.4,
            label=f"{joli(nom)} (AUC = {auc(fpr, tpr):.4f}{suffixe})")
ax.set_xlim(-0.005, 0.2); ax.set_ylim(0.6, 1.005)
ax.set_xlabel('Taux de faux positifs'); ax.set_ylabel('Taux de vrais positifs')
ax.set_title('Courbes ROC sur le test (zoom, positif = Masculin)'); ax.legend(loc='lower right', fontsize=8)
sauver(fig, 'roc_test.png')

fig, ax = plt.subplots(figsize=(4.6, 4))
cm = confusion_matrix(y_te, P['regression_logistique'] > 0.5)
ConfusionMatrixDisplay(cm, display_labels=['Féminin', 'Masculin']).plot(ax=ax, cmap='Blues', colorbar=False)
for i in range(2):
    for j in range(2):
        ax.texts[i * 2 + j].set_text(f"{cm[i, j]}\n({cm[i, j] / cm[i].sum() * 100:.1f} %)")
ax.set_title('Matrice de confusion - régression logistique (test)')
sauver(fig, 'confusion_regression_logistique.png')

## 4. Compromis précision / taux de réponse (mode « incertain »)

In [ ]:
p = P['regression_logistique']
conf, ok = np.maximum(p, 1 - p), (p > 0.5) == y_te
seuils = np.round(np.arange(0.50, 0.96, 0.05), 2)
comp = pd.DataFrame([dict(seuil=s, taux_reponse=(conf >= s).mean(), precision=ok[conf >= s].mean(),
                          noms_a_verifier=int((conf < s).sum())) for s in seuils])
comp.round(4).to_csv(os.path.join(OUT, 'precision_taux_reponse.csv'), index=False)
display(comp.round(4))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(comp['taux_reponse'] * 100, comp['precision'] * 100, marker='o', color=BLEU)
for _, r in comp.iterrows():
    ax.annotate(f"{r['seuil']:.2f}", (r['taux_reponse'] * 100, r['precision'] * 100), fontsize=8,
                xytext=(4, -10), textcoords='offset points')
r = comp.loc[comp['seuil'] == 0.70].iloc[0]
ax.scatter([r['taux_reponse'] * 100], [r['precision'] * 100], s=120, facecolors='none', edgecolors=ROUGE, lw=2,
           label=f"seuil retenu 0,70 : {r['taux_reponse']*100:.1f} % traités, {r['precision']*100:.2f} % de précision")
ax.invert_xaxis(); ax.set_xlabel('Noms traités automatiquement (%)'); ax.set_ylabel('Précision (%)')
ax.set_title('Régression logistique : précision selon le seuil de confiance (test)')
ax.legend(fontsize=8, loc='lower right')
sauver(fig, 'precision_taux_reponse.png')

## 5. Courbe d'apprentissage (validation)

In [ ]:
diag = json.load(open(DIAG_JSON, encoding='utf-8'))
pts = pd.DataFrame(diag['courbe_apprentissage']['points'])
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.errorbar(pts['n_train'], pts['mean'] * 100, yerr=pts['std'].fillna(0) * 100, marker='o', capsize=3, color=BLEU)
ax.set_xlabel("Nombre de noms d'entraînement"); ax.set_ylabel('Accuracy validation (%)')
ax.set_title(f"Courbe d'apprentissage - plafond estimé : {diag['courbe_apprentissage']['plafond_estime']*100:.1f} %")
sauver(fig, 'courbe_apprentissage.png')
pts.round(4).to_csv(os.path.join(OUT, 'courbe_apprentissage.csv'), index=False)

## 6. Interprétabilité du modèle de production (noms fictifs)

Contribution de chaque mot = prédiction du modèle pour ce mot pris seul, ramenée entre
-1 (féminin certain) et +1 (masculin certain). Un nom de famille peut lui aussi porter un genre :
Kouassi, par exemple, est d'abord un prénom akan masculin (garçon né un dimanche).

In [ ]:
m = charger(PAQUET_DIR)
exemples = ['kouassi adjoua marie', 'yao koffi jean marc', 'traore fatoumata', 'akissi grace nadege']
fig, axes = plt.subplots(1, len(exemples), figsize=(3.2 * len(exemples), 3))
for ax, nom in zip(axes, exemples):
    p0 = m.predict_proba([nom])[0]
    mots = nom.split()
    contrib = 2 * m.predict_proba(mots) - 1          # -1 féminin ... +1 masculin
    genre = 'Masculin' if p0 > 0.5 else 'Féminin'
    ax.bar(mots, contrib, color=[BLEU if c > 0 else ROUGE for c in contrib])
    ax.axhline(0, color=GRIS, lw=0.8)
    ax.set_title(f"{nom}\n{genre} ({max(p0, 1-p0)*100:.1f} %)", fontsize=9)
    ax.set_ylim(-1.05, 1.05); ax.tick_params(axis='x', rotation=30)
axes[0].set_ylabel('Contribution du mot seul\n(-1 féminin, +1 masculin)')
fig.suptitle('Interprétabilité par mot (modèle de production)', y=1.05)
sauver(fig, 'interpretabilite_mots.png')

## 7. Configuration retenue pour le v5 (recherche Optuna)

In [ ]:
cfg = json.load(open(BEST_V5))
cles = ['aug', 'n_aug', 'word_drop', 'char_dim', 'word_dim', 'tok_dim', 'n_attn_layers',
        'word_min_freq', 'unk_prob', 'dropout', 'lr', 'batch_size', 'label_smooth']
conf_v5 = pd.DataFrame([(k, cfg[k]) for k in cles if k in cfg], columns=['hyperparamètre', 'valeur'])
conf_v5['valeur'] = conf_v5['valeur'].map(lambda v: f"{v:.4g}" if isinstance(v, float) else v)
conf_v5.to_csv(os.path.join(OUT, 'configuration_v5.csv'), index=False, encoding='utf-8-sig')
display(conf_v5)
print("Fichiers produits :", sorted(os.listdir(OUT)))